In [ ]:
import pandas as pd, numpy as np

# === 路徑 ===
a_path = 'ds_numService_with_billing_onehot.csv'   # A
b_path = 'final_customer_features_dataset.csv'     # B
c_path = 'cleaned_dataset2.csv'                    # C
out_path = 'csr_service_bill.csv'

# === A, B ===
a = pd.read_csv(a_path).rename(columns={'客編': 'CUST_NO'})
b = pd.read_csv(b_path)

# --- B 類別重組 ---
b['MAIN_CATEGORY'] = b['MAIN_CATEGORY'].where(
    b['MAIN_CATEGORY'].isin(b['MAIN_CATEGORY'].value_counts().nlargest(4).index),
    '其他主類')
b['SUB_CATEGORY'] = b['SUB_CATEGORY'].where(
    b['SUB_CATEGORY'].isin(b['SUB_CATEGORY'].value_counts().nlargest(6).index),
    '其他分類')
b_encoded = pd.get_dummies(b[['MAIN_CATEGORY','SUB_CATEGORY']],
                           prefix=['maincat','subcat'], dtype=np.int8)
b = pd.concat([b.drop(columns=['MAIN_CATEGORY','SUB_CATEGORY']), b_encoded], axis=1)

# --- C 使用狀態對照表 (EPON/CM) ---
cols = ['客編','產品名稱','用戶種類','相關編號','起日','迄日',
        '系統台','地區','繳別','使用狀態']
status_map = {}
for chunk in pd.read_csv(c_path, sep='^', engine='python',
                         names=cols, dtype=str, chunksize=300_000):
    sub = chunk[chunk['產品名稱'].isin(['EPON','CM'])]
    sub['status_val'] = np.where(sub['使用狀態']=='使用中', 0, 1)
    status_map.update(sub.groupby('客編')['status_val'].min().to_dict())

# --- 合併 A+B ---
merged = b.merge(a, on='CUST_NO', how='outer')

# --- 補齊使用狀態_數值 ---
mask = merged['使用狀態_數值'].isna()
merged.loc[mask, '使用狀態_數值'] = merged.loc[mask, 'CUST_NO'].map(status_map)

# --- 其餘空值補 0 ---
merged = merged.fillna(0)

# --- 輸出 ---
merged.to_csv(out_path, index=False)
print(f'Done → {out_path} , shape={merged.shape}')


C:\Users\user\AppData\Local\Temp\ipykernel_21288\1146270406.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['status_val'] = np.where(sub['使用狀態']=='使用中', 0, 1)
C:\Users\user\AppData\Local\Temp\ipykernel_21288\1146270406.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['status_val'] = np.where(sub['使用狀態']=='使用中', 0, 1)
C:\Users\user\AppData\Local\Temp\ipykernel_21288\1146270406.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[ro

Done → csr_service_bill.csv , shape=(67249, 39)


In [ ]:
import pandas as pd, numpy as np

# === 路徑 ===
a_path = 'ds_doService_combo_filtered_top10.csv'   # A 工單特徵
b_path = 'simplified_customer_features_top10.csv'  # B 來電記錄特徵
c_path = 'ds_billing_onehot.csv'                    # C 帳單資料特徵


# === A, B ===
a = pd.read_csv(a_path,sep='^').rename(columns={'客編': 'CUST_NO'})
b = pd.read_csv(b_path,sep=',')
c = pd.read_csv(c_path,sep=',').rename(columns={'客編': 'CUST_NO'})

In [9]:
a.head()

,CUST_NO,相關編號,工單日期,30天內,60天內,90天內,90天以上,平均等待天數,產品名稱,用戶種類,...,工單原因_重複率,工單原因_種類數,工單原因_光纖斷_佔比,工單原因_光纖斷_是否發生,工單原因_光纖斷_次數,工單原因_其他_測試正常_次數,工單原因_STB操作不熟導致_次數,工單原因_電視輸入源切錯_是否發生,工單原因_STB操作不熟導致_是否發生,工單原因_電視輸入源切錯_次數
0,1224962,2655785,2024-11-07 15:48:01,1,0,0,2,3.666667,CM,一般收視戶,...,0.667,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1222656,2655799,2024-10-20 14:48:29,1,0,0,0,1.000000,EPON,一般收視戶,...,0.000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,1228353,2655827,2024-10-27 14:57:41,1,0,0,0,1.000000,CM,一般收視戶,...,0.000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1228226,2655846,2024-11-06 12:57:24,1,0,0,1,1.500000,DTV,一般收視戶,...,0.000,2.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,538555,2655860,2024-01-01 15:40:51,1,0,0,0,0.000000,EPON,一般收視戶,...,0.000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
a.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5340 entries, 0 to 5339
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CUST_NO              5340 non-null   int64  
 1   相關編號                 5340 non-null   int64  
 2   工單日期                 5340 non-null   object 
 3   30天內                 5340 non-null   int64  
 4   60天內                 5340 non-null   int64  
 5   90天內                 5340 non-null   int64  
 6   90天以上                5340 non-null   int64  
 7   平均等待天數               5340 non-null   float64
 8   產品名稱                 5340 non-null   object 
 9   用戶種類                 5340 non-null   object 
 10  起日                   5340 non-null   object 
 11  迄日                   0 non-null      float64
 12  系統台                  5340 non-null   object 
 13  地區                   5340 non-null   object 
 14  繳別                   5340 non-null   int64  
 15  使用狀態                 5340 non-null   o

In [13]:
a['產品名稱'].value_counts()

產品名稱
EPON             1888
DTV              1848
CM               1497
CATV               74
LINE TV            23
居家智慧攝影機             7
MESH WIFI加值服務       2
熊搭心(瑪帛)             1
Name: count, dtype: int64

In [14]:
# 篩選a資料集中僅保留產品名稱為EPON或CM的資料
print("=== 篩選前a資料集狀況 ===")
print(f"篩選前a資料集筆數: {len(a)}")
print("產品名稱分布:")
print(a['產品名稱'].value_counts())

# 篩選僅保留EPON或CM的資料
a_filtered = a[a['產品名稱'].isin(['EPON', 'CM'])].copy()

print(f"\n=== 篩選後a資料集狀況 ===")
print(f"篩選後a資料集筆數: {len(a_filtered)}")
print(f"刪除筆數: {len(a) - len(a_filtered)}")
print(f"保留比例: {len(a_filtered) / len(a) * 100:.1f}%")
print("篩選後產品名稱分布:")
print(a_filtered['產品名稱'].value_counts())

# 更新a變數
a = a_filtered.copy()

print(f"\n✅ 已成功篩選a資料集，僅保留產品名稱為EPON或CM的資料")
print(f"更新後a資料集形狀: {a.shape}")

=== 篩選前a資料集狀況 ===
篩選前a資料集筆數: 5340
產品名稱分布:
產品名稱
EPON             1888
DTV              1848
CM               1497
CATV               74
LINE TV            23
居家智慧攝影機             7
MESH WIFI加值服務       2
熊搭心(瑪帛)             1
Name: count, dtype: int64

=== 篩選後a資料集狀況 ===
篩選後a資料集筆數: 3385
刪除筆數: 1955
保留比例: 63.4%
篩選後產品名稱分布:
產品名稱
EPON    1888
CM      1497
Name: count, dtype: int64

✅ 已成功篩選a資料集，僅保留產品名稱為EPON或CM的資料
更新後a資料集形狀: (3385, 29)


In [15]:
# 對產品名稱進行one-hot編碼
print("=== 對產品名稱進行one-hot編碼 ===")

# 顯示編碼前的狀況
print("編碼前產品名稱分布:")
print(a['產品名稱'].value_counts())
print(f"編碼前a資料集形狀: {a.shape}")

# 進行one-hot編碼
product_encoded = pd.get_dummies(a['產品名稱'], prefix='product', dtype=int)

# 將編碼後的欄位合併到原資料集，並移除原始的'產品名稱'欄位
a_encoded = pd.concat([a.drop(columns=['產品名稱']), product_encoded], axis=1)

print(f"\n=== one-hot編碼結果 ===")
print(f"新增的one-hot欄位: {list(product_encoded.columns)}")
print(f"編碼後a資料集形狀: {a_encoded.shape}")
print(f"新增欄位數: {len(product_encoded.columns)}")

# 顯示新增欄位的分布
print(f"\n各one-hot欄位的值分布:")
for col in product_encoded.columns:
    print(f"{col}: {a_encoded[col].sum()} 筆 (比例: {a_encoded[col].mean()*100:.1f}%)")

# 更新a變數
a = a_encoded.copy()

print(f"\n✅ 已成功對產品名稱進行one-hot編碼")
print(f"更新後a資料集形狀: {a.shape}")

=== 對產品名稱進行one-hot編碼 ===
編碼前產品名稱分布:
產品名稱
EPON    1888
CM      1497
Name: count, dtype: int64
編碼前a資料集形狀: (3385, 29)

=== one-hot編碼結果 ===
新增的one-hot欄位: ['product_CM', 'product_EPON']
編碼後a資料集形狀: (3385, 30)
新增欄位數: 2

各one-hot欄位的值分布:
product_CM: 1497 筆 (比例: 44.2%)
product_EPON: 1888 筆 (比例: 55.8%)

✅ 已成功對產品名稱進行one-hot編碼
更新後a資料集形狀: (3385, 30)


In [16]:
a['用戶種類'].value_counts()

用戶種類
一般收視戶    3385
Name: count, dtype: int64

In [17]:
# 刪除a資料集中的指定欄位
print("=== 刪除a資料集中的指定欄位 ===")

# 要刪除的欄位列表
columns_to_drop = ['工單日期', '用戶種類', '起日', '迄日', '系統台', '地區', '繳別']

# 檢查刪除前的狀況
print(f"刪除前a資料集形狀: {a.shape}")
print(f"刪除前總欄位數: {len(a.columns)}")

# 檢查哪些欄位實際存在於資料集中
existing_columns = [col for col in columns_to_drop if col in a.columns]
missing_columns = [col for col in columns_to_drop if col not in a.columns]

print(f"\n=== 欄位檢查結果 ===")
print(f"存在的欄位: {existing_columns}")
if missing_columns:
    print(f"不存在的欄位: {missing_columns}")

# 執行刪除操作（僅刪除存在的欄位）
if existing_columns:
    a_cleaned = a.drop(columns=existing_columns)
    print(f"\n=== 刪除結果 ===")
    print(f"實際刪除的欄位: {existing_columns}")
    print(f"刪除的欄位數: {len(existing_columns)}")
    print(f"刪除後a資料集形狀: {a_cleaned.shape}")
    print(f"刪除後總欄位數: {len(a_cleaned.columns)}")
    
    # 更新a變數
    a = a_cleaned.copy()
    
    print(f"\n✅ 已成功刪除指定欄位")
else:
    print(f"\n⚠️ 沒有找到要刪除的欄位")

=== 刪除a資料集中的指定欄位 ===
刪除前a資料集形狀: (3385, 30)
刪除前總欄位數: 30

=== 欄位檢查結果 ===
存在的欄位: ['工單日期', '用戶種類', '起日', '迄日', '系統台', '地區', '繳別']

=== 刪除結果 ===
實際刪除的欄位: ['工單日期', '用戶種類', '起日', '迄日', '系統台', '地區', '繳別']
刪除的欄位數: 7
刪除後a資料集形狀: (3385, 23)
刪除後總欄位數: 23

✅ 已成功刪除指定欄位


In [18]:
# 顯示刪除後的資料集資訊
print("=== 刪除後資料集資訊 ===")
print(f"當前a資料集形狀: {a.shape}")
print(f"剩餘欄位:")
for i, col in enumerate(a.columns, 1):
    print(f"{i:2d}. {col}")

=== 刪除後資料集資訊 ===
當前a資料集形狀: (3385, 23)
剩餘欄位:
 1. CUST_NO
 2. 相關編號
 3. 30天內
 4. 60天內
 5. 90天內
 6. 90天以上
 7. 平均等待天數
 8. 使用狀態
 9. 工單嚴重程度_平均
10. 工單嚴重程度_最高
11. 工單升級趨勢
12. 工單原因_重複率
13. 工單原因_種類數
14. 工單原因_光纖斷_佔比
15. 工單原因_光纖斷_是否發生
16. 工單原因_光纖斷_次數
17. 工單原因_其他_測試正常_次數
18. 工單原因_STB操作不熟導致_次數
19. 工單原因_電視輸入源切錯_是否發生
20. 工單原因_STB操作不熟導致_是否發生
21. 工單原因_電視輸入源切錯_次數
22. product_CM
23. product_EPON


In [10]:
b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44349 entries, 0 to 44348
Data columns (total 53 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   MAX_SENTIMENT_SCORE        44349 non-null  int64  
 1   PROBLEM_COMPLEXITY_SCORE   44349 non-null  float64
 2   SUB_CAT_方案異動CM轉EOC_COUNT   44349 non-null  float64
 3   AVG_CALL_INTERVAL_DAYS     44349 non-null  float64
 4   MAIN_CAT_非連線問題_COUNT       44349 non-null  float64
 5   SUB_CATEGORY_DIVERSITY     44349 non-null  int64  
 6   MAIN_CAT_新產品_COUNT         44349 non-null  float64
 7   SUB_CAT_建議與抱怨_RATIO        44349 non-null  float64
 8   SUB_CAT_方案異動CM轉EOC_RATIO   44349 non-null  float64
 9   TOTAL_CALLS                44349 non-null  int64  
 10  STD_CALL_INTERVAL_DAYS     44349 non-null  float64
 11  ESCALATION_RISK            44349 non-null  float64
 12  SUB_CAT_建議與抱怨_COUNT        44349 non-null  float64
 13  TOTAL_SENTIMENT_SCORE      44349 non-null  int

In [11]:
c.head()

,CUST_NO,平均繳款延遲日數,paytype_1,paytype_3,paytype_6,paytype_12,paytype_15,paytype_99,paymethod_APP臨櫃代收(7-11),paymethod_CVS臨櫃代收(7-11),paymethod_信用卡扣款,paymethod_其他繳款方式,paymethod_廠商代收,paymethod_當月繳帳單,paymethod_簡訊帳單臨櫃代收(7-11),paymethod_金融機構轉帳
0,895701,-46.700000,0,0,0,1,0,0,0,0,0,1,0,0,0,0
1,1129002,19.333333,1,0,0,0,0,0,0,0,0,1,0,0,0,0
2,223725,13.420000,0,0,0,1,0,0,0,1,0,0,0,0,0,0
3,533302,-43.285714,0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,1026745,-34.800000,0,0,0,1,0,0,0,0,0,0,0,0,1,0


In [19]:
# 以b資料集為基礎進行資料合併
print("=== 以b資料集為基礎進行資料合併 ===")

# 顯示合併前各資料集的狀況
print("=== 合併前各資料集狀況 ===")
print(f"b資料集 (基礎): {b.shape}")
print(f"a資料集 (工單特徵): {a.shape}")
print(f"c資料集 (帳單特徵): {c.shape}")

print(f"\nb資料集CUST_NO數量: {b['CUST_NO'].nunique()}")
print(f"a資料集CUST_NO數量: {a['CUST_NO'].nunique()}")
print(f"c資料集CUST_NO數量: {c['CUST_NO'].nunique()}")

# 檢查重疊的CUST_NO
b_cust_no = set(b['CUST_NO'])
a_cust_no = set(a['CUST_NO'])
c_cust_no = set(c['CUST_NO'])

a_overlap = a_cust_no & b_cust_no
c_overlap = c_cust_no & b_cust_no

print(f"\n=== CUST_NO重疊分析 ===")
print(f"a與b重疊的CUST_NO數量: {len(a_overlap)}")
print(f"c與b重疊的CUST_NO數量: {len(c_overlap)}")
print(f"a與b重疊比例: {len(a_overlap) / len(b_cust_no) * 100:.1f}%")
print(f"c與b重疊比例: {len(c_overlap) / len(b_cust_no) * 100:.1f}%")

=== 以b資料集為基礎進行資料合併 ===
=== 合併前各資料集狀況 ===
b資料集 (基礎): (44349, 53)
a資料集 (工單特徵): (3385, 23)
c資料集 (帳單特徵): (346407, 16)

b資料集CUST_NO數量: 44349
a資料集CUST_NO數量: 3381
c資料集CUST_NO數量: 346407

=== CUST_NO重疊分析 ===
a與b重疊的CUST_NO數量: 3131
c與b重疊的CUST_NO數量: 38046
a與b重疊比例: 7.1%
c與b重疊比例: 85.8%


In [25]:
# 步驟1: 將a資料集中存在於b資料集的CUST_NO合併至b
print("\n=== 步驟1: 合併a資料集 ===")

# 篩選a資料集中存在於b資料集的客戶
a_to_merge = a[a['CUST_NO'].isin(b['CUST_NO'])].copy()
print(f"從a資料集中篩選出可合併的筆數: {len(a_to_merge)}")

# 執行左連接 (以b為主，保留b的所有記錄)
merged_step1 = b.merge(a_to_merge, on='CUST_NO', how='left')
print(f"合併a後的資料集形狀: {merged_step1.shape}")

# 檢查合併後的空值情況
a_columns = [col for col in a_to_merge.columns if col != 'CUST_NO']
print(f"來自a資料集的新欄位數: {len(a_columns)}")
print(f"新欄位中的空值統計:")
for col in a_columns[:5]:  # 顯示前5個欄位的空值情況
    null_count = merged_step1[col].isnull().sum()
    print(f"  {col}: {null_count} 個空值 ({null_count/len(merged_step1)*100:.1f}%)")
if len(a_columns) > 5:
    print(f"  ... 還有 {len(a_columns)-5} 個欄位")


=== 步驟1: 合併a資料集 ===
從a資料集中篩選出可合併的筆數: 3135
合併a後的資料集形狀: (44353, 75)
來自a資料集的新欄位數: 22
新欄位中的空值統計:
  相關編號: 41218 個空值 (92.9%)
  30天內: 41218 個空值 (92.9%)
  60天內: 41218 個空值 (92.9%)
  90天內: 41218 個空值 (92.9%)
  90天以上: 41218 個空值 (92.9%)
  ... 還有 17 個欄位


In [26]:
# 步驟2: 將c資料集中存在於b資料集的CUST_NO合併至已合併的資料
print("\n=== 步驟2: 合併c資料集 ===")

# 篩選c資料集中存在於b資料集的客戶
c_to_merge = c[c['CUST_NO'].isin(b['CUST_NO'])].copy()
print(f"從c資料集中篩選出可合併的筆數: {len(c_to_merge)}")

# 執行左連接 (以step1結果為主)
final_merged = merged_step1.merge(c_to_merge, on='CUST_NO', how='left')
print(f"最終合併後的資料集形狀: {final_merged.shape}")

# 檢查合併後的空值情況
c_columns = [col for col in c_to_merge.columns if col != 'CUST_NO']
print(f"來自c資料集的新欄位數: {len(c_columns)}")
print(f"新欄位中的空值統計:")
for col in c_columns[:5]:  # 顯示前5個欄位的空值情況
    null_count = final_merged[col].isnull().sum()
    print(f"  {col}: {null_count} 個空值 ({null_count/len(final_merged)*100:.1f}%)")
if len(c_columns) > 5:
    print(f"  ... 還有 {len(c_columns)-5} 個欄位")


=== 步驟2: 合併c資料集 ===
從c資料集中篩選出可合併的筆數: 38046
最終合併後的資料集形狀: (44353, 90)
來自c資料集的新欄位數: 15
新欄位中的空值統計:
  平均繳款延遲日數: 6303 個空值 (14.2%)
  paytype_1: 6303 個空值 (14.2%)
  paytype_3: 6303 個空值 (14.2%)
  paytype_6: 6303 個空值 (14.2%)
  paytype_12: 6303 個空值 (14.2%)
  ... 還有 10 個欄位


In [27]:
# 檢查最終合併結果
print("\n=== 最終合併結果摘要 ===")
print(f"原始b資料集: {b.shape}")
print(f"最終合併資料集: {final_merged.shape}")
print(f"新增欄位數: {final_merged.shape[1] - b.shape[1]}")
print(f"客戶數量保持不變: {len(final_merged) == len(b)}")

# 檢查總體空值情況
total_nulls = final_merged.isnull().sum().sum()
total_cells = final_merged.shape[0] * final_merged.shape[1]
print(f"總空值數: {total_nulls:,}")
print(f"總空值比例: {total_nulls/total_cells*100:.2f}%")

# 顯示欄位分組
original_b_cols = list(b.columns)
a_cols = [col for col in final_merged.columns if col in a.columns and col != 'CUST_NO']
c_cols = [col for col in final_merged.columns if col in c.columns and col != 'CUST_NO']

print(f"\n=== 欄位分組 ===")
print(f"原始b資料集欄位: {len(original_b_cols)} 個")
print(f"來自a資料集欄位: {len(a_cols)} 個")
print(f"來自c資料集欄位: {len(c_cols)} 個")
print(f"總欄位數: {final_merged.shape[1]} 個")


=== 最終合併結果摘要 ===
原始b資料集: (44349, 53)
最終合併資料集: (44353, 90)
新增欄位數: 37
客戶數量保持不變: False
總空值數: 1,002,758
總空值比例: 25.12%

=== 欄位分組 ===
原始b資料集欄位: 53 個
來自a資料集欄位: 22 個
來自c資料集欄位: 15 個
總欄位數: 90 個


In [29]:
# final_merged['使用狀態_數值'] = final_merged['使用狀態'].map({'使用中': 0, '未使用': 1})
final_merged['使用狀態'].value_counts()

使用狀態
使用中     2982
停用        69
欠款斷線      43
維修中       32
設備加裝       4
拆機中        4
移機中        1
Name: count, dtype: int64

In [32]:
d_path = 'cleaned_dataset2.csv'                    # C
d = pd.read_csv(d_path,sep='^').rename(columns={'客編': 'CUST_NO'})

In [33]:
d.head()

,CUST_NO,產品名稱,用戶種類,相關編號,起日,迄日,系統台,地區,繳別,使用狀態
0,503,EPON,一般收視戶,2569881,2023/04/20 11:17:58,NaN,大屯,大里區,1,使用中
1,506,CATV,一般收視戶,709,1998/11/01 00:00:00,1998/11/30 00:00:00,大屯,太平區,2,停用
2,511,CATV,一般收視戶,714,2005/07/19 15:50:03,NaN,大屯,太平區,1,停用
3,511,CATV,一般收視戶,713,1997/06/01 00:00:00,1997/08/31 00:00:00,大屯,太平區,3,停用
4,511,DTV,一般收視戶,1262815,2014/06/27 19:07:39,NaN,大屯,太平區,1,停用


In [34]:
# 當final_merged資料集中使用狀態為空值時，從d資料集填補
print("=== 處理final_merged中的使用狀態空值 ===")

# 檢查final_merged中使用狀態的空值情況
if '使用狀態' in final_merged.columns:
    null_count_before = final_merged['使用狀態'].isnull().sum()
    print(f"處理前使用狀態空值數量: {null_count_before}")
    
    if null_count_before > 0:
        # 從d資料集中篩選產品名稱為EPON或CM的資料
        d_filtered = d[d['產品名稱'].isin(['EPON', 'CM'])].copy()
        print(f"d資料集中EPON/CM產品數量: {len(d_filtered)}")
        
        # 建立CUST_NO到使用狀態的對應字典
        # 如果同一個客戶有多筆記錄，取最常見的狀態
        status_mapping = d_filtered.groupby('CUST_NO')['使用狀態'].agg(
            lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]
        ).to_dict()
        
        print(f"從d資料集建立的狀態對應數量: {len(status_mapping)}")
        
        # 找出final_merged中使用狀態為空值的記錄
        null_mask = final_merged['使用狀態'].isnull()
        null_cust_nos = final_merged.loc[null_mask, 'CUST_NO']
        
        print(f"需要填補的客戶數量: {len(null_cust_nos)}")
        
        # 檢查有多少客戶可以從d資料集中找到對應狀態
        available_mappings = [cust_no for cust_no in null_cust_nos if cust_no in status_mapping]
        print(f"可從d資料集找到對應狀態的客戶數量: {len(available_mappings)}")
        
        # 執行填補
        for cust_no in available_mappings:
            final_merged.loc[(final_merged['CUST_NO'] == cust_no) & 
                           (final_merged['使用狀態'].isnull()), '使用狀態'] = status_mapping[cust_no]
        
        # 檢查填補後的結果
        null_count_after = final_merged['使用狀態'].isnull().sum()
        filled_count = null_count_before - null_count_after
        
        print(f"\n=== 填補結果 ===")
        print(f"處理前空值數量: {null_count_before}")
        print(f"處理後空值數量: {null_count_after}")
        print(f"成功填補數量: {filled_count}")
        print(f"填補比例: {filled_count / null_count_before * 100:.1f}%")
        
        # 顯示填補後使用狀態的分布
        print(f"\n=== 填補後使用狀態分布 ===")
        print(final_merged['使用狀態'].value_counts(dropna=False))
        
    else:
        print("使用狀態欄位沒有空值，無需填補")
        
else:
    print("❌ final_merged中找不到'使用狀態'欄位")
    print(f"可用欄位: {list(final_merged.columns)}")

=== 處理final_merged中的使用狀態空值 ===
處理前使用狀態空值數量: 41218
d資料集中EPON/CM產品數量: 342017
從d資料集建立的狀態對應數量: 321993
需要填補的客戶數量: 41218
可從d資料集找到對應狀態的客戶數量: 32201

=== 填補結果 ===
處理前空值數量: 41218
處理後空值數量: 9017
成功填補數量: 32201
填補比例: 78.1%

=== 填補後使用狀態分布 ===
使用狀態
使用中     29899
NaN      9017
停用       3775
欠款斷線     1006
拆機中       332
維修中       205
設備加裝       72
暫停         16
移機中        15
無法完工       13
派收中         3
Name: count, dtype: int64


In [36]:
# 刪除final_merged中使用狀態為空值的記錄
print("=== 刪除使用狀態為空值的記錄 ===")

# 檢查刪除前的狀況
print(f"刪除前final_merged資料集形狀: {final_merged.shape}")

if '使用狀態' in final_merged.columns:
    # 統計空值情況
    null_count = final_merged['使用狀態'].isnull().sum()
    print(f"使用狀態空值數量: {null_count}")
    print(f"空值比例: {null_count / len(final_merged) * 100:.2f}%")
    
    if null_count > 0:
        # 顯示刪除前使用狀態的分布
        print(f"\n=== 刪除前使用狀態分布 ===")
        print(final_merged['使用狀態'].value_counts(dropna=False))
        
        # 刪除使用狀態為空值的記錄
        final_merged_cleaned = final_merged.dropna(subset=['使用狀態']).copy()
        
        print(f"\n=== 刪除結果 ===")
        print(f"刪除前筆數: {len(final_merged)}")
        print(f"刪除後筆數: {len(final_merged_cleaned)}")
        print(f"刪除筆數: {len(final_merged) - len(final_merged_cleaned)}")
        print(f"保留比例: {len(final_merged_cleaned) / len(final_merged) * 100:.2f}%")
        
        # 驗證刪除結果
        remaining_nulls = final_merged_cleaned['使用狀態'].isnull().sum()
        print(f"刪除後使用狀態空值數量: {remaining_nulls}")
        
        # 顯示刪除後使用狀態的分布
        print(f"\n=== 刪除後使用狀態分布 ===")
        print(final_merged_cleaned['使用狀態'].value_counts())
        
        # 更新final_merged變數
        final_merged = final_merged_cleaned.copy()
        
        print(f"\n✅ 已成功刪除使用狀態為空值的記錄")
        print(f"更新後final_merged形狀: {final_merged.shape}")
        
    else:
        print("✅ 使用狀態欄位沒有空值，無需刪除")
        
else:
    print("❌ final_merged中找不到'使用狀態'欄位")
    print(f"可用欄位: {list(final_merged.columns)}")

=== 刪除使用狀態為空值的記錄 ===
刪除前final_merged資料集形狀: (44353, 90)
使用狀態空值數量: 9017
空值比例: 20.33%

=== 刪除前使用狀態分布 ===
使用狀態
使用中     29899
NaN      9017
停用       3775
欠款斷線     1006
拆機中       332
維修中       205
設備加裝       72
暫停         16
移機中        15
無法完工       13
派收中         3
Name: count, dtype: int64

=== 刪除結果 ===
刪除前筆數: 44353
刪除後筆數: 35336
刪除筆數: 9017
保留比例: 79.67%
刪除後使用狀態空值數量: 0

=== 刪除後使用狀態分布 ===
使用狀態
使用中     29899
停用       3775
欠款斷線     1006
拆機中       332
維修中       205
設備加裝       72
暫停         16
移機中        15
無法完工       13
派收中         3
Name: count, dtype: int64

✅ 已成功刪除使用狀態為空值的記錄
更新後final_merged形狀: (35336, 90)


In [37]:
# 在final_merged中新增使用狀態_數值欄位
print("=== 新增使用狀態_數值欄位 ===")

# 檢查使用狀態欄位是否存在
if '使用狀態' in final_merged.columns:
    # 顯示轉換前的使用狀態分布
    print("轉換前使用狀態分布:")
    print(final_merged['使用狀態'].value_counts(dropna=False))
    
    # 創建使用狀態_數值欄位：使用中=0，其他=1
    final_merged['使用狀態_數值'] = final_merged['使用狀態'].apply(
        lambda x: 0 if x == '使用中' else 1
    )
    
    print(f"\n=== 轉換結果 ===")
    print("轉換規則: 使用中 → 0, 其他 → 1")
    
    # 顯示轉換後的數值分布
    print(f"\n使用狀態_數值分布:")
    print(final_merged['使用狀態_數值'].value_counts().sort_index())
    
    # 交叉驗證轉換結果
    print(f"\n=== 轉換驗證 ===")
    cross_tab = pd.crosstab(final_merged['使用狀態'], final_merged['使用狀態_數值'], 
                           margins=True, dropna=False)
    print("使用狀態 vs 使用狀態_數值 交叉表:")
    print(cross_tab)
    
    # 檢查是否有空值
    null_count = final_merged['使用狀態_數值'].isnull().sum()
    if null_count > 0:
        print(f"⚠️ 使用狀態_數值欄位有 {null_count} 個空值")
    else:
        print("✅ 使用狀態_數值欄位無空值")
        
    print(f"\n✅ 已成功新增使用狀態_數值欄位")
    print(f"資料集形狀: {final_merged.shape}")
    
else:
    print("❌ final_merged中找不到'使用狀態'欄位")
    print(f"可用欄位: {list(final_merged.columns)}")

=== 新增使用狀態_數值欄位 ===
轉換前使用狀態分布:
使用狀態
使用中     29899
停用       3775
欠款斷線     1006
拆機中       332
維修中       205
設備加裝       72
暫停         16
移機中        15
無法完工       13
派收中         3
Name: count, dtype: int64

=== 轉換結果 ===
轉換規則: 使用中 → 0, 其他 → 1

使用狀態_數值分布:
使用狀態_數值
0    29899
1     5437
Name: count, dtype: int64

=== 轉換驗證 ===
使用狀態 vs 使用狀態_數值 交叉表:
使用狀態_數值      0     1    All
使用狀態                       
使用中      29899     0  29899
停用           0  3775   3775
拆機中          0   332    332
暫停           0    16     16
欠款斷線         0  1006   1006
派收中          0     3      3
無法完工         0    13     13
移機中          0    15     15
維修中          0   205    205
設備加裝         0    72     72
All      29899  5437  35336
✅ 使用狀態_數值欄位無空值

✅ 已成功新增使用狀態_數值欄位
資料集形狀: (35336, 91)


In [38]:
# 處理空值 (用0填充)
print("\n=== 處理空值 ===")
print("將所有空值填充為0...")

# 填充空值
merged_filled = final_merged.fillna(0)

# 驗證填充結果
remaining_nulls = merged_filled.isnull().sum().sum()
print(f"填充後剩餘空值數: {remaining_nulls}")

if remaining_nulls == 0:
    print("✅ 所有空值已成功填充")
else:
    print("⚠️ 仍有空值存在")

# 更新最終結果
final_dataset = merged_filled.copy()

print(f"\n=== 最終資料集 ===")
print(f"資料集形狀: {final_dataset.shape}")
print(f"空值數量: {final_dataset.isnull().sum().sum()}")


=== 處理空值 ===
將所有空值填充為0...
填充後剩餘空值數: 0
✅ 所有空值已成功填充

=== 最終資料集 ===
資料集形狀: (35336, 91)
空值數量: 0


In [39]:
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35336 entries, 0 to 44352
Data columns (total 91 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   MAX_SENTIMENT_SCORE        35336 non-null  int64  
 1   PROBLEM_COMPLEXITY_SCORE   35336 non-null  float64
 2   SUB_CAT_方案異動CM轉EOC_COUNT   35336 non-null  float64
 3   AVG_CALL_INTERVAL_DAYS     35336 non-null  float64
 4   MAIN_CAT_非連線問題_COUNT       35336 non-null  float64
 5   SUB_CATEGORY_DIVERSITY     35336 non-null  int64  
 6   MAIN_CAT_新產品_COUNT         35336 non-null  float64
 7   SUB_CAT_建議與抱怨_RATIO        35336 non-null  float64
 8   SUB_CAT_方案異動CM轉EOC_RATIO   35336 non-null  float64
 9   TOTAL_CALLS                35336 non-null  int64  
 10  STD_CALL_INTERVAL_DAYS     35336 non-null  float64
 11  ESCALATION_RISK            35336 non-null  float64
 12  SUB_CAT_建議與抱怨_COUNT        35336 non-null  float64
 13  TOTAL_SENTIMENT_SCORE      35336 non-null  int64  


In [40]:
# 重新排序欄位，將基本特徵排在前面
print("=== 重新排序欄位 ===")

# 檢查當前欄位
print(f"排序前final_dataset形狀: {final_dataset.shape}")
print("排序前前10個欄位:")
for i, col in enumerate(final_dataset.columns[:10], 1):
    print(f"{i:2d}. {col}")

# 定義基本特徵欄位的優先順序
basic_features = []

# 1. 客戶編號
if 'CUST_NO' in final_dataset.columns:
    basic_features.append('CUST_NO')

# 2. 使用狀態相關
if '使用狀態' in final_dataset.columns:
    basic_features.append('使用狀態')
if '使用狀態_數值' in final_dataset.columns:
    basic_features.append('使用狀態_數值')

# 3. 產品相關 (product開頭的欄位)
product_cols = [col for col in final_dataset.columns if col.startswith('product_')]
basic_features.extend(sorted(product_cols))

print(f"\n=== 識別出的基本特徵欄位 ===")
for i, col in enumerate(basic_features, 1):
    print(f"{i:2d}. {col}")

# 4. 其他重要特徵（可根據需要調整）
other_important = []
# 主要類別相關
maincat_cols = [col for col in final_dataset.columns if col.startswith('maincat_')]
subcat_cols = [col for col in final_dataset.columns if col.startswith('subcat_')]
other_important.extend(sorted(maincat_cols))
other_important.extend(sorted(subcat_cols))

# 5. 剩餘欄位
remaining_cols = [col for col in final_dataset.columns 
                 if col not in basic_features and col not in other_important]

# 建立新的欄位順序
new_column_order = basic_features + other_important + remaining_cols

print(f"\n=== 新的欄位排序 ===")
print(f"基本特徵: {len(basic_features)} 個")
print(f"其他重要特徵: {len(other_important)} 個") 
print(f"剩餘欄位: {len(remaining_cols)} 個")
print(f"總欄位數: {len(new_column_order)} 個")

=== 重新排序欄位 ===
排序前final_dataset形狀: (35336, 91)
排序前前10個欄位:
 1. MAX_SENTIMENT_SCORE
 2. PROBLEM_COMPLEXITY_SCORE
 3. SUB_CAT_方案異動CM轉EOC_COUNT
 4. AVG_CALL_INTERVAL_DAYS
 5. MAIN_CAT_非連線問題_COUNT
 6. SUB_CATEGORY_DIVERSITY
 7. MAIN_CAT_新產品_COUNT
 8. SUB_CAT_建議與抱怨_RATIO
 9. SUB_CAT_方案異動CM轉EOC_RATIO
10. TOTAL_CALLS

=== 識別出的基本特徵欄位 ===
 1. CUST_NO
 2. 使用狀態
 3. 使用狀態_數值
 4. product_CM
 5. product_EPON

=== 新的欄位排序 ===
基本特徵: 5 個
其他重要特徵: 0 個
剩餘欄位: 86 個
總欄位數: 91 個


In [41]:
# 執行欄位重新排序
print("\n=== 執行欄位重新排序 ===")

# 重新排序資料集
final_dataset_reordered = final_dataset[new_column_order].copy()

print(f"重新排序後的資料集形狀: {final_dataset_reordered.shape}")

# 顯示前20個欄位
print(f"\n=== 排序後前20個欄位 ===")
for i, col in enumerate(final_dataset_reordered.columns[:20], 1):
    print(f"{i:2d}. {col}")

if len(final_dataset_reordered.columns) > 20:
    print(f"... 還有 {len(final_dataset_reordered.columns) - 20} 個欄位")

# 更新final_dataset
final_dataset = final_dataset_reordered.copy()

print(f"\n✅ 已成功重新排序欄位")


=== 執行欄位重新排序 ===
重新排序後的資料集形狀: (35336, 91)

=== 排序後前20個欄位 ===
 1. CUST_NO
 2. 使用狀態
 3. 使用狀態_數值
 4. product_CM
 5. product_EPON
 6. MAX_SENTIMENT_SCORE
 7. PROBLEM_COMPLEXITY_SCORE
 8. SUB_CAT_方案異動CM轉EOC_COUNT
 9. AVG_CALL_INTERVAL_DAYS
10. MAIN_CAT_非連線問題_COUNT
11. SUB_CATEGORY_DIVERSITY
12. MAIN_CAT_新產品_COUNT
13. SUB_CAT_建議與抱怨_RATIO
14. SUB_CAT_方案異動CM轉EOC_RATIO
15. TOTAL_CALLS
16. STD_CALL_INTERVAL_DAYS
17. ESCALATION_RISK
18. SUB_CAT_建議與抱怨_COUNT
19. TOTAL_SENTIMENT_SCORE
20. AVG_MONTHLY_CALLS
... 還有 71 個欄位

✅ 已成功重新排序欄位


In [42]:
# 驗證重新排序結果
print("\n=== 驗證重新排序結果 ===")

# 檢查基本特徵是否都在前面
print("基本特徵欄位位置:")
for col in basic_features:
    if col in final_dataset.columns:
        position = list(final_dataset.columns).index(col) + 1
        print(f"{col}: 第 {position} 位")

# 顯示前幾筆資料預覽（僅顯示前10個欄位）
print(f"\n=== 資料預覽（前10個欄位） ===")
preview_cols = final_dataset.columns[:10]
print("欄位名稱:", list(preview_cols))
print("\n前5筆資料:")
print(final_dataset[preview_cols].head())


=== 驗證重新排序結果 ===
基本特徵欄位位置:
CUST_NO: 第 1 位
使用狀態: 第 2 位
使用狀態_數值: 第 3 位
product_CM: 第 4 位
product_EPON: 第 5 位

=== 資料預覽（前10個欄位） ===
欄位名稱: ['CUST_NO', '使用狀態', '使用狀態_數值', 'product_CM', 'product_EPON', 'MAX_SENTIMENT_SCORE', 'PROBLEM_COMPLEXITY_SCORE', 'SUB_CAT_方案異動CM轉EOC_COUNT', 'AVG_CALL_INTERVAL_DAYS', 'MAIN_CAT_非連線問題_COUNT']

前5筆資料:
   CUST_NO  使用狀態  使用狀態_數值  product_CM  product_EPON  MAX_SENTIMENT_SCORE  \
0   856873   使用中        0         0.0           0.0                    0   
2   288844   使用中        0         0.0           0.0                    3   
3  1254673  欠款斷線        1         1.0           0.0                    2   
4  1224281   使用中        0         0.0           0.0                    1   
5  1242727    停用        1         1.0           0.0                    2   

   PROBLEM_COMPLEXITY_SCORE  SUB_CAT_方案異動CM轉EOC_COUNT  AVG_CALL_INTERVAL_DAYS  \
0                  0.086957                       0.0                1.545455   
2                  0.285714                    

In [43]:
out_path = 'csr_service_bill_enhanced.csv'
final_dataset.to_csv(out_path, index=False)
print(f"✅ 資料已成功輸出至 {out_path}")

✅ 資料已成功輸出至 csr_service_bill_enhanced.csv
